In [5]:
import numpy as np
import random

# ==========================================
# Grid World
# ==========================================

rows = 4
cols = 4

start_state = (0, 0)
goal_state = (3, 3)

# Obstacles
obstacles = [(1, 1), (2, 1)]

# Actions
# 0 = Up
# 1 = Down
# 2 = Left
# 3 = Right

actions = 4

# ==========================================
# State Mapping
# ==========================================

def state_number(position):
    return position[0] * cols + position[1]

num_states = rows * cols

Q = np.zeros((num_states, actions))

# ==========================================
# Hyperparameters
# ==========================================

learning_rate = 0.8
discount_factor = 0.9

epsilon = 1.0
epsilon_decay = 0.995
epsilon_min = 0.01

episodes = 1000

# ==========================================
# Environment Function
# ==========================================

def step(state, action):

    r, c = state

    if action == 0:      # Up
        r -= 1

    elif action == 1:    # Down
        r += 1

    elif action == 2:    # Left
        c -= 1

    elif action == 3:    # Right
        c += 1

    # Boundary check
    r = max(0, min(rows - 1, r))
    c = max(0, min(cols - 1, c))

    next_state = (r, c)

    # Obstacle
    if next_state in obstacles:
        return state, -100, True

    # Goal
    if next_state == goal_state:
        return next_state, 100, True

    # Normal move
    return next_state, -1, False

# ==========================================
# Training
# ==========================================

rewards = []

for episode in range(episodes):

    state = start_state
    total_reward = 0
    done = False

    while not done:

        state_idx = state_number(state)

        # Epsilon-Greedy
        if random.uniform(0, 1) < epsilon:
            action = random.randint(0, 3)
        else:
            action = np.argmax(Q[state_idx])

        next_state, reward, done = step(state, action)

        next_idx = state_number(next_state)

        # Q-Learning Update
        Q[state_idx, action] = Q[state_idx, action] + learning_rate * (
            reward +
            discount_factor * np.max(Q[next_idx]) -
            Q[state_idx, action]
        )

        state = next_state
        total_reward += reward

    rewards.append(total_reward)

    if epsilon > epsilon_min:
        epsilon *= epsilon_decay

# ==========================================
# Print Q-Table
# ==========================================

print("Q-Table\n")
print(np.round(Q, 2))

# ==========================================
# Learned Path
# ==========================================

print("\nOptimal Path")

state = start_state
path = [state]

while state != goal_state:

    action = np.argmax(Q[state_number(state)])

    next_state, reward, done = step(state, action)

    if next_state == state:
        break

    path.append(next_state)
    state = next_state

    if len(path) > 20:
        break

print(path)

# ==========================================
# Display Grid
# ==========================================

grid = np.full((rows, cols), ".")

for obs in obstacles:
    grid[obs] = "X"

grid[start_state] = "S"
grid[goal_state] = "G"

print("\nGrid")
for row in grid:
    print(" ".join(row))

Q-Table

[[ 48.46  54.95  48.46  54.95]
 [ 54.95 -44.05  48.46  62.17]
 [ 62.17  70.19  54.95  70.19]
 [ 70.19  79.1   62.14  70.19]
 [ 48.46  62.17  54.95 -44.13]
 [  0.     0.     0.     0.  ]
 [ 62.17  79.1  -28.81  79.1 ]
 [ 70.19  89.    70.19  79.1 ]
 [ 54.95  70.19  62.17 -36.83]
 [  0.     0.     0.     0.  ]
 [ 70.08  89.   -19.9   88.99]
 [ 79.1  100.    79.1   89.  ]
 [ 62.17  70.19  70.19  79.1 ]
 [-19.9   79.1   70.19  89.  ]
 [ 79.1   89.    79.1  100.  ]
 [  0.     0.     0.     0.  ]]

Optimal Path
[(0, 0), (1, 0), (2, 0), (3, 0), (3, 1), (3, 2), (3, 3)]

Grid
S . . .
. X . .
. X . .
. . . G


In [3]:
pip install pandas nltk scikit-learn transformers torch sentence-transformers scipy

  Using cached transformers-5.12.1-py3-none-any.whl.metadata (33 kB)
  Using cached sentence_transformers-5.6.0-py3-none-any.whl.metadata (18 kB)
  Using cached huggingface_hub-1.21.0-py3-none-any.whl.metadata (14 kB)
  Using cached regex-2026.6.28-cp313-cp313-win_amd64.whl.metadata (41 kB)
  Using cached tokenizers-0.22.2-cp39-abi3-win_amd64.whl.metadata (7.4 kB)
  Using cached safetensors-0.8.0-cp310-abi3-win_amd64.whl.metadata (4.2 kB)
  Using cached click-8.4.2-py3-none-any.whl.metadata (2.6 kB)
  Using cached hf_xet-1.5.1-cp37-abi3-win_amd64.whl.metadata (4.9 kB)
Using cached transformers-5.12.1-py3-none-any.whl (11.2 MB)
Using cached huggingface_hub-1.21.0-py3-none-any.whl (721 kB)
   ---------------------------------------- 0.0/4.0 MB ? eta -:--:--
   -- ------------------------------------- 0.3/4.0 MB ? eta -:--:--
   ------------- -------------------------- 1.3/4.0 MB 5.5 MB/s eta 0:00:01
   -------------------- ------------------- 2.1/4.0 MB 4.6 MB/s eta 0:00:01
   ----------

  You can safely remove it manually.
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
streamlit 1.51.0 requires protobuf<7,>=3.20, but you have protobuf 7.35.1 which is incompatible.


In [6]:
import nltk
from nltk.stem import PorterStemmer, WordNetLemmatizer
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords

from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

from gensim.models import Word2Vec

# Download NLTK data (only first time)
nltk.download('punkt')
nltk.download('stopwords')
nltk.download('wordnet')

# ---------------------------------
# Sample Sentences
# ---------------------------------

sentences = [
    "Machine learning is fascinating.",
    "Natural language processing is a part of artificial intelligence.",
    "Deep learning uses neural networks."
]

# ---------------------------------
# Tokenization
# ---------------------------------

tokens = [word_tokenize(sentence.lower()) for sentence in sentences]

print("Tokenized Sentences:")
print(tokens)

# ---------------------------------
# Remove Stopwords
# ---------------------------------

stop_words = set(stopwords.words('english'))

filtered_tokens = []

for sentence in tokens:
    filtered = [word for word in sentence if word.isalpha() and word not in stop_words]
    filtered_tokens.append(filtered)

print("\nAfter Removing Stopwords:")
print(filtered_tokens)

# ---------------------------------
# Stemming
# ---------------------------------

stemmer = PorterStemmer()

stemmed = []

for sentence in filtered_tokens:
    stemmed.append([stemmer.stem(word) for word in sentence])

print("\nStemmed Words:")
print(stemmed)

# ---------------------------------
# Lemmatization
# ---------------------------------

lemmatizer = WordNetLemmatizer()

lemmatized = []

for sentence in filtered_tokens:
    lemmatized.append([lemmatizer.lemmatize(word) for word in sentence])

print("\nLemmatized Words:")
print(lemmatized)

# ---------------------------------
# Bag of Words
# ---------------------------------

vectorizer = CountVectorizer()

bow = vectorizer.fit_transform(sentences)

print("\nBag of Words Vocabulary:")
print(vectorizer.get_feature_names_out())

print("\nBag of Words Matrix:")
print(bow.toarray())

# ---------------------------------
# TF-IDF
# ---------------------------------

tfidf = TfidfVectorizer()

tfidf_matrix = tfidf.fit_transform(sentences)

print("\nTF-IDF Vocabulary:")
print(tfidf.get_feature_names_out())

print("\nTF-IDF Matrix:")
print(tfidf_matrix.toarray())

# ---------------------------------
# Word2Vec
# ---------------------------------

model = Word2Vec(
    sentences=filtered_tokens,
    vector_size=50,
    window=3,
    min_count=1,
    workers=1
)

print("\nWord2Vec Vector for 'learning':")

print(model.wv['learning'])

print("\nMost Similar Words to 'learning':")

print(model.wv.most_similar('learning'))

Tokenized Sentences:
[['machine', 'learning', 'is', 'fascinating', '.'], ['natural', 'language', 'processing', 'is', 'a', 'part', 'of', 'artificial', 'intelligence', '.'], ['deep', 'learning', 'uses', 'neural', 'networks', '.']]

After Removing Stopwords:
[['machine', 'learning', 'fascinating'], ['natural', 'language', 'processing', 'part', 'artificial', 'intelligence'], ['deep', 'learning', 'uses', 'neural', 'networks']]

Stemmed Words:
[['machin', 'learn', 'fascin'], ['natur', 'languag', 'process', 'part', 'artifici', 'intellig'], ['deep', 'learn', 'use', 'neural', 'network']]

Lemmatized Words:
[['machine', 'learning', 'fascinating'], ['natural', 'language', 'processing', 'part', 'artificial', 'intelligence'], ['deep', 'learning', 'us', 'neural', 'network']]

Bag of Words Vocabulary:
['artificial' 'deep' 'fascinating' 'intelligence' 'is' 'language'
 'learning' 'machine' 'natural' 'networks' 'neural' 'of' 'part'
 'processing' 'uses']

Bag of Words Matrix:
[[0 0 1 0 1 0 1 1 0 0 0 0 0 

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\SruthinJayakumaran\AppData\Roaming\nltk_data.
[nltk_data]     ..
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\SruthinJayakumaran\AppData\Roaming\nltk_data.
[nltk_data]     ..
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\SruthinJayakumaran\AppData\Roaming\nltk_data.
[nltk_data]     ..
[nltk_data]   Package wordnet is already up-to-date!
